In [1]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(funcName)s - %(message)s"
)
load_dotenv()

True

In [28]:
system_prompt = """
You are an expert problem-solving assistant. When given a question or problem statement, follow this structured workflow:

## Workflow

### Step 1 — Decompose
Call the `create_subtasks` tool to break the problem into clear, logical, and sequential sub-tasks before attempting any solution.
(e.g. for the question How long will it take to drive 300 km at an average speed of 60 km/h?, to solve the question the list of subtasks will be:
[Identify the formula for travel time, Determine the given distance, Determine the given speed, Apply the formula time = distance / speed, Compute the final travel time]

### Step 2 — Solve Iteratively
Work through each sub-task one by one:
- Reason carefully and produce a thorough solution for the current sub-task.
- After completing each sub-task, call the `cross_completed` tool.
- The `cross_completed` tool will **re-render the full task list** after every iteration, where:
  - Completed sub-tasks are shown with ~~strikethrough~~ text.
  - Pending sub-tasks remain as normal text.
  - If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
- Continue until all sub-tasks are marked as complete.

### Step 3 — Display Final Results
Once all sub-tasks are resolved, render a final rich console output in the terminal using the `rich` library that includes:
- **Sub-task breakdown**: Each sub-task with its solution, displayed in a structured panel or table.
- **Overall solution**: A synthesized, comprehensive answer derived from all sub-task results, highlighted clearly.

## Output Requirements
- Use `rich` console formatting: panels, rules, syntax highlighting, and color-coded text where appropriate.
- Keep sub-task solutions concise but complete.
- The overall solution must be a coherent synthesis — not just a list of sub-task answers.
- Always follow the order: `create_subtasks` → solve → `cross_completed` → solve → `cross_completed` → ...(until all sub-tasks completerd)... → rich display.

"""

In [20]:
todos=[]
completed=[]

In [21]:
def show(text):
    try: 
        Console.print(text)
    except:
        print(text)

In [22]:
def show_todos(sentences:list[str], index:int):
    result=''
    for i, sentence in enumerate(sentences):
        if i<(index):
            result+=f"Todo #{i+1}: [green][strike]{sentence}[/strike][/green]\n"
        else:
            result+=f"Todo #{i+1}: {sentence}"
    show(result)
    

In [23]:
def create_subtasks(subtasks_list:list[str]):

    todos.append(subtasks_list)

    show_todos(subtasks_list, 0)
    return todos

create_subtasks_json={
    'type':'function',
    'function': {
        'name': 'create_subtasks',
        'description': 'using the list of sub_tasks, adding them to a new list of todos and displaying the subtasks',
        'parameters':{
            'type':'object',
            'properties': {
                'subtasks_list':{
                    'type':'array',
                    'items': {'type':'string'},
                    'description': 'the list of divided subtasks, (e.g. for the question How long will it take to drive 300 km at an average speed of 60 km/h?, to solve the question the list of subtasks will be [Identify the formula for travel time, Determine the given distance, Determine the given speed, Apply the formula time = distance / speed, Compute the final travel time]'
                }
            },
            'required':['subtasks_list']
        }
    }
}

In [24]:
def cross_completed(sentences:list[str], index:int):

    solved=[]
    not_solved=[]
    for i, sentence in enumerate(sentences):
        if i < index:
            solved.append(sentence)
        else:
            not_solved.append(sentence)

    dict_completed={
        'Solved':solved,
        'Not Solved': not_solved
    }
    show_todos(sentences, index)
    return dict_completed

cross_completed_json={
    'type':'function',
    'function':{
        'name':'cross_completed',
        'description': 'to send the dictionary containing the solved and the unsolved sentences after each solved subtask',
        'parameters': {
            'type':'object',
            'properties':{
                'sentences': {
                    'type': 'array',
                    'items': {'type':'string'},
                    'description': 'list of the subtasks'
                },
                'index':{
                    'type': 'integer',
                    'description': 'the number of solved subtasks , (for e.g. if out of 5 total subtasks, 4 are solved then the value of index will be 4)'
                }
            },
            'required':['sentences', 'index']
        }
    }
}

In [25]:
def tool_calls_func(tool_calls:list):
    results=[]
    for tool_call in tool_calls:
        tool_name=tool_call.function.name
        function=globals().get(tool_name)
        argument=json.loads(tool_call.function.arguments)
        response=function(**argument)
        results.append((response, tool_name))
    return results    


In [29]:
gemini_key=os.getenv('GOOGLE_GEMINI')

client=OpenAI(api_key=gemini_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/") 

tools=[create_subtasks_json, cross_completed_json]

def loop(message):
    messages=[{'role':'system', 'content': system_prompt}]
    messages.append({'role':'user', 'content': message})

    done=False
    while not done:
        try:
            logging.info(f'Calling LLM Call')
            llm_tool=client.chat.completions.create(
                model="gemini-2.5-flash-lite",
                messages=messages,
                tools=tools,
                tool_choice="auto"
            )
            logging.info(f'LLM Output -- {llm_tool}')
            assistant_message=llm_tool.choices[0].message

            if assistant_message.tool_calls:
                tool_calls=assistant_message.tool_calls
                
                logging.info({f'Calling tool {tool_calls}'})
                responses=tool_calls_func(tool_calls=tool_calls)

                for response in responses:
                    messages.append({'role':'assistant', 'content':f'Tool Call {response[1]} result: {response[0]}'})
                    logging.info(f'{response}')
            
            else:
                show(assistant_message.content)
                done=True
        except Exception as e:
            logging.error(e)
            raise


In [30]:
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
loop(user_message)

2026-03-08 17:29:04,676 - INFO - loop - Calling LLM Call
2026-03-08 17:29:06,281 - INFO - _send_single_request - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-08 17:29:06,281 - INFO - loop - LLM Output -- ChatCompletion(id='imStaf21D7PXg8UP1cCYqAI', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='function-call-10249623353586363623', function=Function(arguments='{"subtasks_list":["Determine the distance between Boston and New York","Calculate the distance traveled by the Boston train by 3:00 pm","Set up an equation to represent the relative distance between the two trains","Calculate the time it takes for the trains to meet after 3:00 pm","Determine the meeting time of the two trains"]}', name='create_subtasks'), type='

Todo #1: Determine the distance between Boston and New YorkTodo #2: Calculate the distance traveled by the Boston train by 3:00 pmTodo #3: Set up an equation to represent the relative distance between the two trainsTodo #4: Calculate the time it takes for the trains to meet after 3:00 pmTodo #5: Determine the meeting time of the two trains


2026-03-08 17:29:08,244 - INFO - _send_single_request - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-08 17:29:08,244 - INFO - loop - LLM Output -- ChatCompletion(id='jGStafm2D77NjuMP7OKf4Aw', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\nSubtasks:\n1. Determine the distance between Boston and New York (assume D miles)\n2. Calculate the distance the first train travels before the second train departs.\n3. Determine the remaining distance between the two trains when the second train departs.\n4. Calculate the relative speed of the two trains.\n5. Calculate the time it takes for the trains to meet after the second train departs.\n6. Determine the actual meeting time.\n7. Determine the distance between Boston and New York\n8. Calculate the distance traveled by the Boston train by 3:00 pm\n9. Set up an equation to represent the relative distance between the two t


Subtasks:
1. Determine the distance between Boston and New York (assume D miles)
2. Calculate the distance the first train travels before the second train departs.
3. Determine the remaining distance between the two trains when the second train departs.
4. Calculate the relative speed of the two trains.
5. Calculate the time it takes for the trains to meet after the second train departs.
6. Determine the actual meeting time.
7. Determine the distance between Boston and New York
8. Calculate the distance traveled by the Boston train by 3:00 pm
9. Set up an equation to represent the relative distance between the two trains
10. Calculate the time it takes for the trains to meet after 3:00 pm
11. Determine the meeting time of the two trains

I will start by determining the distance between Boston and New York. Since this information is not provided, I will assume a distance D.

Subtasks:
1. ~~Determine the distance between Boston and New York (assume D miles)~~
2. Calculate the distance t